In [41]:
import os
import cv2
import numpy as np
import shutil
from pathlib import Path
import random
from collections import Counter

In [42]:

INPUT_DIR = r'E:\VS CODE Coding files\AI_Projects\DCGAN_Custom\Anime_Faces_Dataset_Org\data'
OUTPUT_DIR = r'E:\VS CODE Coding files\AI_Projects\DCGAN_Custom\anime_faces_labeled_new'


In [44]:
# Strict and non-overlapping Hair color ranges in HSV
HAIR_COLOR_RANGES = {
    'red': [
        ([0, 120, 120], [8, 255, 255]),       # Pure red (strict)
        ([172, 120, 120], [180, 255, 255]),   # Red upper range (strict)
        ([155, 80, 150], [172, 255, 255])     # Pink/Magenta (clear boundary)
    ],
    'yellow': [
        ([22, 80, 140], [32, 255, 255])       # Yellow/Blonde (clear gap from red)
    ],
    'green': [
        ([45, 70, 70], [75, 255, 255])        # Green (clear boundaries)
    ],
    'blue': [
        ([95, 100, 100], [120, 255, 255]),    # Pure blue (very strict)
        ([85, 80, 130], [95, 220, 255])       # Cyan/Light blue (strict)
    ],
    'black': [
        ([0, 0, 0], [180, 255, 70]),          # Very dark colors (strict)
        ([0, 0, 70], [180, 25, 140])          # Grey (very low saturation only)
    ]
}

In [45]:
def get_dominant_hair_color(image_path, min_pixel_threshold=800):
    """
    Image se dominant hair color detect karta hai with STRICT accuracy
    """
    try:
        img = cv2.imread(image_path)
        if img is None:
            return None
        
        # Sirf hair region ko analyze karo
        hair_region = get_hair_region(img)
        
        # BGR to HSV conversion
        hsv = cv2.cvtColor(hair_region, cv2.COLOR_BGR2HSV)
        
        # Color detection with confidence scoring
        color_scores = {}
        
        for color_name, ranges in HAIR_COLOR_RANGES.items():
            mask = np.zeros(hsv.shape[:2], dtype=np.uint8)
            
            for lower, upper in ranges:
                lower_bound = np.array(lower)
                upper_bound = np.array(upper)
                temp_mask = cv2.inRange(hsv, lower_bound, upper_bound)
                mask = cv2.bitwise_or(mask, temp_mask)
            
            # Pixel count
            pixel_count = cv2.countNonZero(mask)
            
            # Calculate confidence (percentage of hair region)
            total_pixels = hsv.shape[0] * hsv.shape[1]
            confidence = (pixel_count / total_pixels) * 100
            
            color_scores[color_name] = {
                'pixels': pixel_count,
                'confidence': confidence
            }
        
        # Sort by pixel count
        sorted_colors = sorted(color_scores.items(), 
                              key=lambda x: x[1]['pixels'], 
                              reverse=True)
        
        # Best color
        best_color = sorted_colors[0][0]
        best_pixels = sorted_colors[0][1]['pixels']
        best_confidence = sorted_colors[0][1]['confidence']
        
        # STRICT minimum threshold check
        if best_pixels < min_pixel_threshold:
            return None
        
        # STRICT second best check (clear winner chahiye)
        if len(sorted_colors) > 1:
            second_best_pixels = sorted_colors[1][1]['pixels']
            # Winner ko 2x zyada pixels chahiye (stricter)
            if best_pixels < second_best_pixels * 2.0:
                return None
        
        # STRICT confidence threshold
        if best_confidence < 5.0:  # At least 5% of region should be that color
            return None
        
        return best_color
        
    except Exception as e:
        print(f"Error processing {image_path}: {str(e)}")
        return None

In [46]:
def create_labeled_dataset(input_dir, output_dir, process_all=True):
    """
    Labeled dataset banata hai with improved accuracy
    """
    # Output directories banao
    os.makedirs(output_dir, exist_ok=True)
    for color in HAIR_COLOR_RANGES.keys():
        os.makedirs(os.path.join(output_dir, color), exist_ok=True)
    
    # Sab images ke paths collect karo
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp']
    all_images = []
    
    for ext in image_extensions:
        all_images.extend(Path(input_dir).rglob(f'*{ext}'))
        all_images.extend(Path(input_dir).rglob(f'*{ext.upper()}'))
    
    print(f"Total images found: {len(all_images)}")
    
    if process_all:
        selected_images = all_images
        print(f"Processing ALL {len(selected_images)} images...")
    else:
        num_images = 10000
        if len(all_images) > num_images:
            selected_images = random.sample(all_images, num_images)
        else:
            selected_images = all_images
        print(f"Processing {len(selected_images)} images...")
    
    # Statistics
    color_counts = Counter()
    processed = 0
    skipped = 0
    
    # Har image ko process karo
    for idx, img_path in enumerate(selected_images):
        if (idx + 1) % 500 == 0:
            print(f"Processed: {idx + 1}/{len(selected_images)} | Classified: {processed} | Skipped: {skipped}")
        
        # Hair color detect karo
        hair_color = get_dominant_hair_color(str(img_path))
        
        if hair_color:
            # Unique filename banao
            base_name = f"{hair_color}_{processed:05d}{img_path.suffix}"
            dest_path = os.path.join(output_dir, hair_color, base_name)
            
            # Image copy karo
            shutil.copy2(str(img_path), dest_path)
            
            color_counts[hair_color] += 1
            processed += 1
        else:
            skipped += 1
    
    # Results print karo
    print("\n" + "="*60)
    print("Dataset Creation Complete!")
    print("="*60)
    print(f"\nTotal processed: {processed}")
    print(f"Skipped (ambiguous/unclear): {skipped}")
    print(f"Success rate: {(processed/(processed+skipped)*100):.2f}%")
    print("\nColor Distribution:")
    for color, count in color_counts.most_common():
        percentage = (count/processed)*100
        print(f"  {color.capitalize()}: {count} images ({percentage:.1f}%)")
    print("\n" + "="*60)
    
    return color_counts

In [47]:
if __name__ == "__main__":
    print("="*60)
    print("IMPROVED Anime Faces Hair Color Classification")
    print("="*60)
    print(f"Input directory: {INPUT_DIR}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"Mode: Processing ALL images\n")
    
    # Check input directory
    if not os.path.exists(INPUT_DIR):
        print(f"❌ ERROR: Input directory not found: {INPUT_DIR}")
        print("Please check the path!")
    else:
        print("✅ Input directory found!")
        
        # Confirm before processing all images
        print("\n⚠️  This will process ALL images in the dataset.")
        print("This may take 30-60 minutes depending on your system.")
        
        response = input("\nDo you want to continue? (yes/no): ").strip().lower()
        
        if response in ['yes', 'y']:
            # Dataset banao
            color_distribution = create_labeled_dataset(INPUT_DIR, OUTPUT_DIR, process_all=True)
            
            print(f"\n✅ Dataset successfully created at: {OUTPUT_DIR}")
            print("\nNext steps:")
            print("1. Check each folder for accuracy")
            print("2. Remove any incorrect images manually if needed")
            print("3. Use this dataset for training your model")
        else:
            print("\n❌ Operation cancelled by user.")

IMPROVED Anime Faces Hair Color Classification
Input directory: E:\VS CODE Coding files\AI_Projects\DCGAN_Custom\Anime_Faces_Dataset_Org\data
Output directory: E:\VS CODE Coding files\AI_Projects\DCGAN_Custom\anime_faces_labeled_new
Mode: Processing ALL images

✅ Input directory found!

⚠️  This will process ALL images in the dataset.
This may take 30-60 minutes depending on your system.
Total images found: 86204
Processing ALL 86204 images...
Processed: 500/86204 | Classified: 93 | Skipped: 406
Processed: 1000/86204 | Classified: 199 | Skipped: 800
Processed: 1500/86204 | Classified: 300 | Skipped: 1199
Processed: 2000/86204 | Classified: 402 | Skipped: 1597
Processed: 2500/86204 | Classified: 490 | Skipped: 2009
Processed: 3000/86204 | Classified: 589 | Skipped: 2410
Processed: 3500/86204 | Classified: 675 | Skipped: 2824
Processed: 4000/86204 | Classified: 762 | Skipped: 3237
Processed: 4500/86204 | Classified: 856 | Skipped: 3643
Processed: 5000/86204 | Classified: 949 | Skipped: 4